<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/toy_model/ToyModelEntityRelMultiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
!pip install transformer_lens

In [24]:
import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example(num_relations: int, *, allow_self_loops: bool = False):
    while True: # Keep trying until a valid example is produced
        facts = []
        seen_head_rel = set()

        # Step 1: Build initial unique facts
        while len(facts) < num_relations:
            e = int(rng.integers(0, E))
            t = int(TYPES[rng.integers(0, T)])

            if (e, t) in seen_head_rel:
                continue

            e2 = int(rng.integers(0, E))
            while not allow_self_loops and e2 == e:
                e2 = int(rng.integers(0, E))

            seen_head_rel.add((e, t))
            facts.append((e, t, e2))

        # Step 2: Pick query fact
        q_idx = int(rng.integers(0, num_relations))
        Eq, Tq, E2q = facts[q_idx]
        non_query_indices = [i for i in range(num_relations) if i != q_idx]

        # Step 3: Pick 1-2 random, distinct indices for Eq and Tq insertion
        num_eq_insertions = rng.integers(1, 3) # 1 or 2 insertions for Eq
        num_tq_insertions = rng.integers(1, 3) # 1 or 2 insertions for Tq

        if len(non_query_indices) < num_eq_insertions + num_tq_insertions:
             continue # Not enough non-query facts to satisfy insertions

        # Ensure indices for Eq and Tq insertions are distinct
        all_insert_indices = rng.choice(non_query_indices, size=num_eq_insertions + num_tq_insertions, replace=False)
        eq_insert_indices = all_insert_indices[:num_eq_insertions]
        tq_insert_indices = all_insert_indices[num_eq_insertions:]

        # Step 4: Modify facts for Eq and Tq insertion while preserving (e, t) uniqueness
        successful_insertions = True

        # Eq insertions
        for idx in eq_insert_indices:
            e, t, e2 = facts[idx]
            if (Eq, t) not in seen_head_rel:
                facts[idx] = (Eq, t, e2)
                seen_head_rel.discard((e, t))
                seen_head_rel.add((Eq, t))
            else:
                successful_insertions = False
                break # Couldn't insert without duplicate (e,t), regenerate facts

        if not successful_insertions:
            continue # Regenerate facts

        # Tq insertions
        for idx in tq_insert_indices:
            e, t, e2 = facts[idx]
            if (e, Tq) not in seen_head_rel:
                facts[idx] = (e, Tq, e2)
                seen_head_rel.discard((e, t))
                seen_head_rel.add((e, Tq))
            else:
                successful_insertions = False
                break # Couldn't insert without duplicate (e,t), regenerate facts

        if not successful_insertions:
            continue # Regenerate facts

        # Step 5: Ensure E2q appears in non-query facts (at least once)
        # This part can remain similar to the original ensure_in_non_query logic for a single occurrence
        def ensure_e2q_in_non_query(value, position):
            """Ensure E2q appears at given position in some non-query fact at least once."""
            if not any(f[position] == value for i, f in enumerate(facts) if i != q_idx):
                 # try random order of indices
                shuffled_indices = rng.permutation(non_query_indices)
                for idx in shuffled_indices:
                    e, t, e2 = facts[idx]
                    new_fact = list(facts[idx])
                    old_et = (e, t)
                    new_fact[position] = value
                    new_e, new_t, _ = new_fact

                    # check if (e, t) pair already exists in other non-query facts
                    # and is not the fact we are modifying
                    is_duplicate = False
                    for i, f in enumerate(facts):
                        if i != idx and (f[0], f[1]) == (new_e, new_t):
                            is_duplicate = True
                            break

                    if not is_duplicate:
                        # Temporarily remove old (e,t) and add new (e,t) to seen_head_rel for check
                        temp_seen_head_rel = set(seen_head_rel)
                        if position in [0, 1]:
                            temp_seen_head_rel.discard(old_et)
                            if (new_e, new_t) in temp_seen_head_rel:
                                continue # Still a duplicate with other facts

                        # If no duplicate, commit the change
                        facts[idx] = tuple(new_fact)
                        if position in [0, 1]:
                            seen_head_rel.discard(old_et)
                            seen_head_rel.add((new_e, new_t))
                        return True # Successfully inserted

                return False # Couldn't insert without creating duplicate (e,t)
            return True # Value already present


        if not ensure_e2q_in_non_query(E2q, 2):
             continue # Regenerate facts if E2q cannot be inserted

        # Step 6: Build sequence
        seq = []
        for (e, t, e2) in facts:
            seq.extend([e, t, e2, SEP])
        seq.extend([Tq, Eq, Q])

        label = E2q
        return seq, label, facts


# --- Build dataset ---
rows = []
for _ in tqdm(range(N_WORLDS)):
    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label, _ = produce_example(k, allow_self_loops=False)
    rows.append({"tokens": seq, "label": label})

df = pd.DataFrame(rows)

100%|██████████| 80000/80000 [00:13<00:00, 6104.99it/s]


In [25]:
",".join([str(i) for i in df.iloc[1].tokens]), df.iloc[1].label

('38,106,70,110,38,108,13,110,57,108,84,110,52,108,13,110,108,38,111', 13)

In [26]:
",".join([str(i) for i in df.iloc[2].tokens]), df.iloc[2].label

('18,105,23,110,78,100,41,110,18,107,74,110,71,101,13,110,18,101,13,110,72,101,92,110,101,18,111',
 13)

In [27]:
df

,tokens,label
0,"[63, 105, 26, 110, 30, 105, 7, 110, 1, 101, 81...",26
1,"[38, 106, 70, 110, 38, 108, 13, 110, 57, 108, ...",13
2,"[18, 105, 23, 110, 78, 100, 41, 110, 18, 107, ...",13
3,"[20, 105, 76, 110, 25, 108, 21, 110, 20, 108, ...",21
4,"[13, 107, 92, 110, 91, 102, 12, 110, 59, 100, ...",92
...,...,...
79995,"[76, 108, 5, 110, 76, 107, 16, 110, 38, 105, 8...",16
79996,"[43, 102, 74, 110, 32, 102, 85, 110, 75, 109, ...",56
79997,"[8, 109, 98, 110, 48, 109, 91, 110, 87, 108, 4...",91
79998,"[20, 109, 53, 110, 36, 109, 53, 110, 29, 100, ...",53


In [28]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

In [29]:
IGNORE_INDEX = -100

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [30]:
def train_collate_fn(batch, rng=np.random.default_rng()):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = shuffle_facts(seq.tolist() if torch.is_tensor(seq) else seq, rng)
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def val_collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = seq.tolist() if torch.is_tensor(seq) else seq
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target


In [31]:
def shuffle_facts(seq, rng=np.random.default_rng()):
    seps = [i for i,t in enumerate(seq) if t == SEP]
    context = seq[:seps[-1]+1]
    query_part = seq[seps[-1]+1:]
    facts = [context[i:i+4] for i in range(0, len(context), 4)]
    rng.shuffle(facts)

    return [x for f in facts for x in f] + query_part

In [32]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = EntityBindingDataset(train_df)
val_dataset = EntityBindingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_collate_fn)

# Save dataframes
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

In [33]:
len(train_dataset)

64000

In [34]:
def compute_accuracy(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int = IGNORE_INDEX):
    """
    Accuracy over positions where targets != ignore_index.
    Returns correct_count and total_count
    """
    with torch.no_grad():
        mask = targets.ne(ignore_index)
        total = mask.sum().item()
        preds = logits.argmax(dim=-1)
        correct = preds.masked_select(mask).eq(targets.masked_select(mask)).sum().item()
        return correct, total

In [44]:
import math
import itertools
import torch
import torch.nn as nn
from transformer_lens import HookedTransformer, HookedTransformerConfig
from torch.utils.tensorboard import SummaryWriter

# hparams

LAYERS = [3]
HEADS  = [1]

d_model = 32#256
d_mlp   = 1024
n_ctx   = 64
lr = 5e-4
betas = (0.9, 0.98)
weight_decay = 0
num_epochs = 100

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        d_mlp=d_mlp,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

In [45]:
def grid_search(n_layers: int, n_heads: int):
    # Fresh seeds per run for comparability
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    model = build_model(n_layers, n_heads)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=betas, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    run_name = f"entity_binding_L{n_layers}_H{n_heads}"
    writer = SummaryWriter(f"runs/{run_name}")

    global_step = 0
    best_val_acc = -1.0
    best_val_loss = float('inf') # Initialize best validation loss
    epochs_no_improve = 0 # Counter for epochs without improvement
    early_stop_patience = 10 # Number of epochs to wait for improvement

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        total_train_correct = 0
        total_train_count = 0
        for input_tokens, targets in train_loader:
            input_tokens, targets = input_tokens.to(device), targets.to(device)

            optimizer.zero_grad()
            logits = model(input_tokens)  # shape: [B, T, d_vocab]
            loss = criterion(logits.view(-1, logits.size(-1)),
                              targets.view(-1))
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            c, n = compute_accuracy(logits, targets)
            total_train_correct += c
            total_train_count += n
            step_acc = (c / n) if n else 0.0
            writer.add_scalar("Loss/train", loss.item(), global_step)
            writer.add_scalar("Acc/train_step", step_acc, global_step)
            global_step += 1

        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = (total_train_correct / total_train_count) if total_train_count else 0.0
        print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f}")

        model.eval()
        total_val_loss = 0
        total_val_correct = 0
        total_val_count = 0
        with torch.no_grad():
            for input_tokens, targets in val_loader:
                input_tokens, targets = input_tokens.to(device), targets.to(device)
                logits = model(input_tokens)
                loss = criterion(logits.view(-1, logits.size(-1)),
                              targets.view(-1))
                total_val_loss += loss.item()
                c, n = compute_accuracy(logits, targets)
                total_val_correct += c
                total_val_count += n

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0
        print(f"[Epoch {epoch+1}] Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}")
        writer.add_scalar("Loss/val", avg_val_loss, global_step)
        writer.add_scalar("Acc/val", val_acc, global_step)

        # Early stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc = val_acc # Update best accuracy as well
            epochs_no_improve = 0
            # Optionally save the best model here
            # torch.save(model.state_dict(), f"best_model_L{n_layers}_H{n_heads}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stop_patience:
                print(f"Early stopping triggered after {epoch+1} epochs due to no improvement in validation loss for {early_stop_patience} epochs.")
                # break # Exit the training loop

    writer.close()
    print("Training complete.")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model": model,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "best_val_acc": float(best_val_acc),
        "best_val_epoch": int(epoch - epochs_no_improve) if epochs_no_improve > 0 else epoch, # Adjust best epoch if early stopped
        "stopped_early": epochs_no_improve >= early_stop_patience
    }

In [46]:
results = []
for n_layers, n_heads in itertools.product(LAYERS, HEADS):
    print(f"##### Running config: n_layers={n_layers}, n_heads={n_heads} #####")
    out = grid_search(n_layers, n_heads)
    results.append(out)

##### Running config: n_layers=3, n_heads=1 #####
Moving model to device:  cuda
[Epoch 1] Train Loss: 2.3266 | Train Acc: 0.530
[Epoch 1] Val Loss: 0.8058 | Val Acc: 0.819
[Epoch 2] Train Loss: 0.6061 | Train Acc: 0.855
[Epoch 2] Val Loss: 0.5232 | Val Acc: 0.874
[Epoch 3] Train Loss: 0.4464 | Train Acc: 0.890
[Epoch 3] Val Loss: 0.4228 | Val Acc: 0.893
[Epoch 4] Train Loss: 0.3666 | Train Acc: 0.908
[Epoch 4] Val Loss: 0.4051 | Val Acc: 0.900
[Epoch 5] Train Loss: 0.3029 | Train Acc: 0.924
[Epoch 5] Val Loss: 0.2926 | Val Acc: 0.929
[Epoch 6] Train Loss: 0.2375 | Train Acc: 0.945
[Epoch 6] Val Loss: 0.2279 | Val Acc: 0.949
[Epoch 7] Train Loss: 0.2002 | Train Acc: 0.954
[Epoch 7] Val Loss: 0.2523 | Val Acc: 0.947
[Epoch 8] Train Loss: 0.1810 | Train Acc: 0.958
[Epoch 8] Val Loss: 0.2099 | Val Acc: 0.956
[Epoch 9] Train Loss: 0.1636 | Train Acc: 0.962
[Epoch 9] Val Loss: 0.1797 | Val Acc: 0.959
[Epoch 10] Train Loss: 0.1547 | Train Acc: 0.963
[Epoch 10] Val Loss: 0.1752 | Val Acc: 0.96

In [47]:
# Save the model
torch.save(results[0]['model'].state_dict(), "entity_binding_model.pth")


In [40]:

# Load the model
# Create a new model instance with the same configuration
loaded_model = HookedTransformer(cfg)
loaded_model.load_state_dict(torch.load("entity_binding_model.pth"))
loaded_model = loaded_model.to(device)

print("Model saved and loaded successfully.")

NameError: name 'model' is not defined

In [ ]:

loaded_model.eval()
total_val_loss = 0
total_val_correct = 0
total_val_count = 0
with torch.no_grad():
    for input_tokens, targets in val_loader:
        input_tokens, targets = input_tokens.to(device), targets.to(device)
        logits = loaded_model(input_tokens)
        loss = criterion(logits.view(-1, logits.size(-1)),
                      targets.view(-1))
        total_val_loss += loss.item()
        c, n = compute_accuracy(logits, targets)
        total_val_correct += c
        total_val_count += n

avg_val_loss = total_val_loss / len(val_loader)
val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0

In [ ]:
val_acc, avg_val_loss